# Stage 2b — Cloud Detector: Pipeline-Native Features

**Goal**: Replace the pre-batched `.npy` path (Stage 2a) with features extracted directly from
L1 Zarr stores produced by the ingest pipeline (`pa-convert` → `pa-calibrate`).

**Key improvement over Stage 2a**: The original batch builder sampled the movie-mode stream at
**10-second** intervals; Stage 2a's `extract_cloud_features` hardcoded a 60-second cadence,
discarding that resolution. Stage 2b uses `feature_cadence_s: 10.0` to recover it.

**Label bridge**: Legacy labels are keyed by `feature_uid` (SHA1).  
`pa-prep-cloud-labels` reads the batch ZIPs in-memory and produces a
`(module, t_start_ns, t_end_ns, label)` interval CSV that `pa-features-cloud` consumes directly.

## Workflow
```
§0  Configure paths
§1  Convert legacy labels  →  cloud_labels_v2.csv
§2  Inspect L1 stores       →  verify coverage overlap
§3  Extract features        →  cloud_features_v2.zarr  (pa-features-cloud)
§4  Train                   →  cloud_detector_v2.pt    (pa-train-cloud on RAL cluster)
§5  Evaluate & compare      →  AP, PR curve vs v1
```

**Prerequisites**
- L1 `dp_img16` Zarr stores produced by `nextflow run . --steps ingest`
- RAL Ray cluster running (`ray up conf/ray/ral_cluster_baremetal.yaml`)
- `WANDB_API_KEY` in environment (or set `WANDB_PROJECT = None`)

## §0 — Configuration

In [ ]:
import os
from pathlib import Path

import numpy as np
import xarray as xr

REPO_ROOT     = Path('../../../').resolve()
LABELS_ZIP    = REPO_ROOT / 'assets/models/cloud-detection-training/training-labels.zip'
DATA_ZIP      = REPO_ROOT / 'assets/models/cloud-detection-training/training-data-batch.zip'
BEEGFS_ROOT   = Path('/mnt/beegfs/panoseti_analysis')

# ── Label CSV (output of §1) ──────────────────────────────────────────────────
LABEL_CSV     = REPO_ROOT / 'ml/cloud-detection/cache/cloud_labels_v2.csv'

# ── L1 img16 Zarr stores from the ingest pipeline ────────────────────────────
# Adjust glob pattern to match your --outdir from `nextflow run . --steps ingest`
L1_STORES     = sorted(BEEGFS_ROOT.glob('runs/*/L1/*.dp_img16.*.zarr'))

# ── Feature cache (output of §3) ─────────────────────────────────────────────
CACHE_ZARR    = REPO_ROOT / 'ml/cloud-detection/cache/cloud_features_v2.zarr'
OUT_DIR       = REPO_ROOT / 'ml/cloud-detection/models'
RECIPE_PATH   = REPO_ROOT / 'ml/cloud-detection/recipes/cloud_v2.yml'

# ── Training options ──────────────────────────────────────────────────────────
BATCH_IDS     = list(range(8))   # all 8 batches
HALF_WINDOW_S = 5.0              # label interval half-width (matches 10s sampling cadence)
WANDB_PROJECT = 'panoseti-cloud-detection'

# ── Legacy model for comparison (from Stage 2a) ───────────────────────────────
V2_MODEL_PT   = OUT_DIR / 'cloud_detector_v2.pt'   # from notebook 02

print(f"Labels ZIP   : {LABELS_ZIP.exists()}  ({LABELS_ZIP.stat().st_size/1e6:.0f} MB)")
print(f"Data ZIP     : {DATA_ZIP.exists()}  ({DATA_ZIP.stat().st_size/1e6:.0f} MB)")
print(f"L1 stores    : {len(L1_STORES)} found")
print(f"Label CSV    : {LABEL_CSV.exists()} (will build if missing)")
print(f"Cache Zarr   : {CACHE_ZARR.exists()} (will build if missing)")
print(f"Recipe       : {RECIPE_PATH.exists()}")

## §1 — Convert Legacy Labels → Interval CSV

Reads the batch ZIPs **in-memory** (no BeeGFS extraction), joins
`feature_uid → pano_uid → (module_id, frame_unix_t)`, and writes
`(module, t_start_ns, t_end_ns, label)` to `LABEL_CSV`.

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format='%(message)s')

from panoseti_analysis.adapters.cloud.prep_labels import build_interval_label_csv

if LABEL_CSV.exists():
    print(f"Label CSV already exists: {LABEL_CSV}")
    print("Delete it to force rebuild.")
else:
    print(f"Building interval label CSV from {len(BATCH_IDS)} batches …")
    build_interval_label_csv(
        labels_zip=LABELS_ZIP,
        data_zip=DATA_ZIP,
        out_path=LABEL_CSV,
        batch_ids=BATCH_IDS,
        half_window_s=HALF_WINDOW_S,
        skip_unsure=True,
    )
    print("Done.")

In [ ]:
import pandas as pd

labels_df = pd.read_csv(LABEL_CSV)
print(f"Intervals  : {len(labels_df):,}")
print(f"Cloudy (1) : {(labels_df.label==1).sum():,}")
print(f"Clear  (0) : {(labels_df.label==0).sum():,}")
print(f"Modules    : {sorted(labels_df.module.unique())}")
t0 = labels_df.t_start_ns.min() / 1e9
t1 = labels_df.t_end_ns.max() / 1e9
print(f"Time span  : {t1 - t0:.0f} s  ({(t1-t0)/3600:.2f} h)")
labels_df.head()

## §2 — Inspect L1 Stores

Verify that the L1 store timestamps overlap with the label CSV.

In [ ]:
if not L1_STORES:
    print("WARNING: no L1 stores found.  Run the ingest pipeline first:")
    print("  nextflow run . -profile ral --steps ingest --input samplesheet.csv --outdir /mnt/beegfs/panoseti_analysis/runs/")
else:
    label_t0_ns = labels_df.t_start_ns.min()
    label_t1_ns = labels_df.t_end_ns.max()

    print(f"{'Store':<60} {'module':<8} {'t_start':>22} {'t_end':>22} {'overlap'}")
    print("-" * 120)
    for p in L1_STORES:
        ds = xr.open_zarr(p, consolidated=False)
        if 'median_subtracted' not in ds:
            continue
        module = ds.attrs.get('module', '?')
        t_ns = ds['unix_t_ns'].values
        overlap = not (t_ns[-1] < label_t0_ns or t_ns[0] > label_t1_ns)
        print(f"{p.name:<60} {str(module):<8} {t_ns[0]:>22d} {t_ns[-1]:>22d} {'YES' if overlap else 'NO'}")

## §3 — Extract Features

Runs `pa-features-cloud` on the L1 stores at **10-second cadence** with the interval label CSV.
Writes `CACHE_ZARR` with schema `X(N,2,32,32)`, `t_ns`, `label`, `split`, `module`.

In [ ]:
from panoseti_analysis.adapters.features import run_features_cloud

if CACHE_ZARR.exists():
    print(f"Feature cache already exists: {CACHE_ZARR}")
    print("Delete it to force rebuild.")
elif not L1_STORES:
    print("ERROR: No L1 stores available — cannot extract features.")
else:
    print(f"Extracting features from {len(L1_STORES)} L1 stores at 10s cadence …")
    run_features_cloud(
        stores_list=L1_STORES,
        out_path=CACHE_ZARR,
        recipe_path=RECIPE_PATH,
        label_csv=LABEL_CSV,
    )
    print("Done.")

In [ ]:
import matplotlib.pyplot as plt

ds = xr.open_zarr(CACHE_ZARR, consolidated=False)
X_all   = ds['X'].values.astype(np.float32)       # (N, 2, 32, 32)
y_all   = ds['label'].values.astype(np.int64)      # (N,)
split   = ds['split'].values                        # (N,) str
t_ns    = ds['t_ns'].values                         # (N,) int64
modules = ds['module'].values                       # (N,) str

N = len(y_all)
labeled = y_all >= 0
print(f"Total samples : {N:,}")
print(f"Labeled       : {labeled.sum():,}  (unlabeled: {(~labeled).sum():,})")
print(f"Cloudy  (1)   : {(y_all==1).sum():,}")
print(f"Clear   (0)   : {(y_all==0).sum():,}")
print(f"Train   split : {(split=='train').sum():,}")
print(f"Val     split : {(split=='val').sum():,}")
print(f"Test    split : {(split=='test').sum():,}")
print(f"Cadence check : median Δt = {np.median(np.diff(t_ns))/1e9:.1f} s  (expected ~10 s)")

fig, axes = plt.subplots(1, 3, figsize=(12, 3))
axes[0].bar(['clear','cloudy'], [(y_all==0).sum(),(y_all==1).sum()], color=['steelblue','tomato'])
axes[0].set_title('Label distribution')
axes[1].pie([(split==s).sum() for s in ['train','val','test']],
            labels=['train','val','test'], autopct='%1.0f%%',
            colors=['steelblue','orange','tomato'])
axes[1].set_title('Train/val/test split')
axes[2].hist(np.diff(t_ns)/1e9, bins=30, color='steelblue')
axes[2].set_xlabel('Δt between windows (s)')
axes[2].set_title('Cadence distribution')
plt.tight_layout()
plt.show()

## §4 — Train

Same training infrastructure as notebook 02.  Attaches to the RAL Ray cluster.

In [ ]:
import ray
from panoseti_analysis.adapters.ray.train_cloud import run_train_cloud

if WANDB_PROJECT and os.environ.get('WANDB_API_KEY'):
    print(f"W&B tracking enabled: project='{WANDB_PROJECT}'")
else:
    print("W&B not configured — using TensorBoard.")
    os.environ['WANDB_MODE'] = 'disabled'

if ray.is_initialized():
    ray.shutdown()

_training_runtime_env = {
    "working_dir": str(REPO_ROOT),
    "env_vars": {"PYTHONPATH": "src"},
    "excludes": [
        ".venv/", ".git/", "*.zarr/", "uv.lock", ".claude/",
        "assets/models/cloud-detection-training/",
        "ml/*/cache/", "ml/*/models/", "ml/*/data/",
        "pypff/example/",
        "grpc/src/panoseti_grpc/daq_data/simulated_data_dir/",
        "grpc/scripts/daq_data/simulated_data_dir/",
    ],
}
if os.environ.get('WANDB_API_KEY'):
    _training_runtime_env["env_vars"]["WANDB_API_KEY"] = os.environ['WANDB_API_KEY']

ray.init(address="auto", runtime_env=_training_runtime_env, ignore_reinit_error=True)
print(f"Ray connected: {ray.cluster_resources()}")

OUT_DIR.mkdir(parents=True, exist_ok=True)

pt_path, json_path, prov_path = run_train_cloud(
    feature_cache=CACHE_ZARR,
    out_dir=OUT_DIR,
    recipe_path=RECIPE_PATH,
    launcher='attach',
    local_cache_dir=None,
    wandb_project=WANDB_PROJECT,
    lineage_out=None,
)
print(f"Model saved : {pt_path}")
print(f"Bundle      : {json_path}")

## §5 — Evaluate & Compare

Compares the v2 pipeline-native model (this notebook) against the v2-legacy model (notebook 02).

In [ ]:
import torch
from sklearn.metrics import (
    average_precision_score, confusion_matrix, ConfusionMatrixDisplay,
    precision_recall_curve,
)
from panoseti_analysis.algorithms.cloud_detector import CloudDetection, extract_cloud_features
from panoseti_analysis.config.models import CloudInferParams

def _eval_on_val(model_path: Path, X_val: np.ndarray, y_val: np.ndarray) -> tuple:
    model = CloudDetection()
    model.load_state_dict(torch.load(model_path, weights_only=True, map_location='cpu'))
    model.eval()
    with torch.no_grad():
        logits = model(torch.from_numpy(X_val))
        probs = torch.softmax(logits, dim=1)[:, 1].numpy()
    preds = (probs >= 0.5).astype(np.int64)
    acc = (preds == y_val).mean()
    ap = average_precision_score(y_val, probs)
    return probs, preds, acc, ap

ds = xr.open_zarr(CACHE_ZARR, consolidated=False)
X_all = ds['X'].values.astype(np.float32)
y_all = ds['label'].values.astype(np.int64)
split = ds['split'].values

val_mask = (split == 'val') & (y_all >= 0)
X_val, y_val = X_all[val_mask], y_all[val_mask]

# Evaluate current model (§4 output)
new_model_pt = sorted(OUT_DIR.glob('cloud_detector_v2*.pt'))[-1] if any(OUT_DIR.glob('cloud_detector_v2*.pt')) else pt_path
probs_new, preds_new, acc_new, ap_new = _eval_on_val(new_model_pt, X_val, y_val)

print(f"Pipeline model  — acc={acc_new:.3f}  AP={ap_new:.3f}")

# Compare against Stage 2a legacy model (if available on same val set — note: feature space matches)
if V2_MODEL_PT.exists():
    probs_leg, preds_leg, acc_leg, ap_leg = _eval_on_val(V2_MODEL_PT, X_val, y_val)
    print(f"Legacy model    — acc={acc_leg:.3f}  AP={ap_leg:.3f}")
else:
    print(f"Legacy model not found at {V2_MODEL_PT} — skipping comparison.")
    probs_leg = None

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Confusion matrix
cm = confusion_matrix(y_val, preds_new)
ConfusionMatrixDisplay(cm, display_labels=['clear','cloudy']).plot(ax=axes[0], colorbar=False)
axes[0].set_title(f'Confusion matrix (val)\nAcc={acc_new:.3f}  AP={ap_new:.3f}')

# Score distribution
axes[1].hist(probs_new[y_val==0], bins=20, alpha=0.7, color='steelblue', label='clear (true)')
axes[1].hist(probs_new[y_val==1], bins=20, alpha=0.7, color='tomato', label='cloudy (true)')
axes[1].axvline(0.5, color='black', linestyle='--')
axes[1].set_xlabel('P(cloudy)')
axes[1].set_title('Score distribution (val)')
axes[1].legend(fontsize=8)

# PR curves
prec_new, rec_new, _ = precision_recall_curve(y_val, probs_new)
axes[2].plot(rec_new, prec_new, color='steelblue', label=f'Pipeline v2 (AP={ap_new:.3f})')
if probs_leg is not None:
    prec_leg, rec_leg, _ = precision_recall_curve(y_val, probs_leg)
    axes[2].plot(rec_leg, prec_leg, color='tomato', linestyle='--', label=f'Legacy v2 (AP={ap_leg:.3f})')
axes[2].set_xlabel('Recall')
axes[2].set_ylabel('Precision')
axes[2].set_title('Precision-Recall (val, cloudy class)')
axes[2].legend(fontsize=8)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()